# Classification metrics

Looking at one metric to decide if a models output is good is not enough, in classification you count the amount of times you classidied correctly that it is that thing and the amount of times you said that it is not that thing correctly, you also count the amount of times you said it was that thing incorrectly and the amount of times you said that it is not that thing incorrectly. These are called True positive and True negative, False positive and False negative respectively. 
1. From these values you can calculate metrics now one of the most used ones is accuracy, which The amout of correct prediction divided by the amount of predictions (TP+TN/TP+TF+FP+FN) this alone is not enough because high accuracy with an imbalaned dataset (900 class a, 100 class b) accuracy could be 90% but all the class b entries are incorrect, so accuracy alone is not enough.
2. Precision calculates the amount of positive entries you got right out of all the entries the model predicted as positive (TP/TP+FP) this also has issues because it does not give the entire story because it gives no information about the TN, and FN so FN could be very high and you would have no idea.
3. Recall calculates the amount of positive entries the model got right out of all the positive entries there are (TP/TP+FN) which also leaves out information from both TN and FP so it alone does not give the full picture.
4. F1 is a metric that combines Precision and Recall although it has multiple calculations inside of it, even if you count it as one it also is not enough because it leaves out the TN classifications which could be very low.

To get the full picture you can use all these calculations (metrics) to get a more clear idea and which ones to pay attention to depends on what is more important to you, for example if in a model FP is very bad to you, you would sacrifice FN being high so that it has as low of FP count as possible. These depend on the situation.

In [5]:
# Evaluate Logistic Regression using the earlier custom NLP components.
from pathlib import Path
import sys

import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Import the preprocessor and tokenizer from week 02 and vectorizer from week 06.
current_path = Path.cwd().resolve()
repository_root = next(
    (path for path in (current_path, *current_path.parents)
     if (path / "phase-1-machine-learning-nlp").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

phase_1 = repository_root / "phase-1-machine-learning-nlp"
part_02 = phase_1 / "02-preprocessing-tokenization"
part_06 = phase_1 / "06-vectorization"
sys.path.insert(0, str(part_02 / "src"))
sys.path.insert(0, str(part_06 / "src"))

from preprocessor import preprocess_text
from tokenizer import regex_tokenize
from vectorizer import CustomTfidfVectorizer


def make_model():
    """Create the same vectorizer and Logistic Regression pipeline."""
    return Pipeline([
        ("tfidf", CustomTfidfVectorizer(
            preprocessor=preprocess_text, tokenizer=regex_tokenize,
            min_df=5, max_df=0.95, max_features=30_000,
        )),
        ("logistic_regression", LogisticRegression(
            C=4.0, solver="liblinear", max_iter=1_000, random_state=42
        )),
    ])


def get_metrics(labels, predictions, probabilities):
    return pd.Series({
        "accuracy": accuracy_score(labels, predictions),
        "positive precision": precision_score(labels, predictions),
        "positive recall": recall_score(labels, predictions),
        "positive F1": f1_score(labels, predictions),
        "ROC-AUC": roc_auc_score(labels, probabilities),
    })


imdb = pd.read_csv(part_02 / "data" / "IMDB Dataset.csv")
imdb = imdb.dropna(subset=["review", "sentiment"])
imdb = imdb.drop_duplicates(subset="review").reset_index(drop=True)
imdb["label"] = imdb["sentiment"].map({"negative": 0, "positive": 1})

X_train, X_test, y_train, y_test = train_test_split(
    imdb["review"], imdb["label"], test_size=0.20,
    random_state=42, stratify=imdb["label"],
)

model = make_model()
model.fit(X_train, y_train)
test_predictions = model.predict(X_test)
test_probabilities = model.predict_proba(X_test)[:, 1]

print("Test metrics:")
display(get_metrics(y_test, test_predictions, test_probabilities)
        .to_frame("value").style.format("{:.4f}"))
print("\nConfusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, test_predictions),
    index=["actual negative", "actual positive"],
    columns=["predicted negative", "predicted positive"],
))
print("\nClassification report:")
print(classification_report(
    y_test, test_predictions, target_names=["negative", "positive"], digits=4
))


Test metrics:


,value
accuracy,0.8875
positive precision,0.8796
positive recall,0.8987
positive F1,0.8891
ROC-AUC,0.9536



Confusion matrix:


,predicted negative,predicted positive
actual negative,4328,612
actual positive,504,4473



Classification report:
              precision    recall  f1-score   support

    negative     0.8957    0.8761    0.8858      4940
    positive     0.8796    0.8987    0.8891      4977

    accuracy                         0.8875      9917
   macro avg     0.8877    0.8874    0.8874      9917
weighted avg     0.8876    0.8875    0.8874      9917



200 
100 1
100 0

70 1 -> 60 1 , 10 0
130 0 -> 90 0 , 40 1

# try to explain the metrics in words using examples

In [6]:
# Remove half of the positive training examples and train the same pipeline again.
# The vectorizer is refitted only on the reduced training set; the test set stays fixed.
positive_indexes = y_train[y_train == 1].index
positive_indexes_to_remove = positive_indexes.to_series().sample(
    n=len(positive_indexes) // 2, random_state=42
).to_numpy()
X_train_imbalanced = X_train.drop(index=positive_indexes_to_remove)
y_train_imbalanced = y_train.drop(index=positive_indexes_to_remove)

imbalanced_model = clone(model)
imbalanced_model.fit(X_train_imbalanced, y_train_imbalanced)
imbalanced_predictions = imbalanced_model.predict(X_test)
imbalanced_probabilities = imbalanced_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "original model": get_metrics(
        y_test, test_predictions, test_probabilities
    ),
    "half positive examples": get_metrics(
        y_test, imbalanced_predictions, imbalanced_probabilities
    ),
})
comparison["change"] = (
    comparison["half positive examples"] - comparison["original model"]
)

print("Training class counts before and after removing positive examples:")
display(pd.DataFrame({
    "original": y_train.value_counts().reindex([0, 1]),
    "after removal": y_train_imbalanced.value_counts().reindex([0, 1]),
}, index=[0, 1]).rename(index={0: "negative", 1: "positive"}))
print("\nMetric changes, with emphasis on the positive class:")
display(comparison.style.format("{:.4f}"))
print("\nClassification report after removing half of the positive examples:")
print(classification_report(
    y_test, imbalanced_predictions,
    target_names=["negative", "positive"], digits=4
))


Training class counts before and after removing positive examples:


,original,after removal
negative,19758,19758
positive,19907,9954



Metric changes, with emphasis on the positive class:


,original model,half positive examples,change
accuracy,0.8875,0.8248,-0.0626
positive precision,0.8796,0.9530,0.0734
positive recall,0.8987,0.6847,-0.2140
positive F1,0.8891,0.7969,-0.0922
ROC-AUC,0.9536,0.9497,-0.0039



Classification report after removing half of the positive examples:
              precision    recall  f1-score   support

    negative     0.7526    0.9660    0.8460      4940
    positive     0.9530    0.6847    0.7969      4977

    accuracy                         0.8248      9917
   macro avg     0.8528    0.8254    0.8215      9917
weighted avg     0.8532    0.8248    0.8214      9917

